In [1]:
!pip uninstall -y -q torchvision torchaudio
!pip install -q -U transformers accelerate bitsandbytes youtube-transcript-api streamlit
print('Done. Now RESTART the session/kernel, then continue from the next cell.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 88.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 88.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 30.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.29 requires torchvision, which is not installed.
Done. Now RESTART the session/kernel, then continue from the next cell.


In [2]:
%%writefile summarizer.py
import re

# Instruct LLM (Qwen2.5) instead of BART-CNN: more accurate, no made-up sentences, supports Arabic output.

MODELS = {
    "Qwen2.5-1.5B (fast, lower quality)": ("Qwen/Qwen2.5-1.5B-Instruct", False),
    "Qwen2.5-3B (recommended)": ("Qwen/Qwen2.5-3B-Instruct", False),
    "Qwen2.5-7B 4-bit (best quality)": ("Qwen/Qwen2.5-7B-Instruct", True),
}
DEFAULT_MODEL_LABEL = "Qwen2.5-3B (recommended)"
LANGUAGES = ["English", "Arabic", "French", "Spanish", "German"]

SYSTEM_PROMPT = (
    "You are a careful assistant that summarizes YouTube video transcripts. "
    "The transcript is auto-generated speech-to-text: it has no speaker labels, "
    "names may be misspelled, and it contains filler words and jokes. "
    "Use ONLY information that is stated in the text. Never invent names, numbers, "
    "places, channels, websites or sources. "
    "Ignore sponsor reads, 'subscribe' requests and small talk unless they matter to the main story."
)

STYLE_INSTRUCTIONS = {
    "short": "Write ONE paragraph of 3-4 sentences: what the video is about, who takes part, and how it ends.",
    "standard": (
        "Use exactly this format:\n"
        "### Overview\n2-3 sentences: what the video is, who takes part, and the main idea.\n"
        "### Key points\n5-8 bullet points with the most important events, claims or steps, in order.\n"
        "### Outcome\n1-3 sentences: the final result, the key numbers, and any twist or conclusion."
    ),
    "detailed": (
        "Use exactly this format:\n"
        "### Overview\n2-3 sentences: what the video is, who takes part, and the main idea.\n"
        "### What happens (in order)\n8-12 bullet points covering the whole video from start to end.\n"
        "### Key numbers and facts\nBullet points with the important numbers, names and rules.\n"
        "### Outcome\n2-4 sentences: the final result and any twist or conclusion."
    ),
}
MAX_NEW_TOKENS = {"short": 220, "standard": 650, "detailed": 1000}


# ---------------------------------------------------------------- transcript
def extract_video_id(url):
    patterns = [
        r"youtu\.be/([0-9A-Za-z_-]{11})",
        r"embed/([0-9A-Za-z_-]{11})",
        r"shorts/([0-9A-Za-z_-]{11})",
        r"[?&]v=([0-9A-Za-z_-]{11})",
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None


def get_youtube_transcript(video_url, languages=None):
    from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound

    languages = languages or ["en"]
    video_id = extract_video_id(video_url)
    if not video_id:
        raise ValueError("Could not extract the video ID from this URL.")

    if hasattr(YouTubeTranscriptApi, "fetch"):
        api = YouTubeTranscriptApi()
        try:
            fetched = api.fetch(video_id, languages=languages)
        except NoTranscriptFound:
            # requested language not available -> take any available transcript
            fetched = next(iter(api.list(video_id))).fetch()
        return " ".join(s.text for s in fetched)

    # old API
    try:
        data = YouTubeTranscriptApi.get_transcript(video_id, languages=languages)
    except NoTranscriptFound:
        data = next(iter(YouTubeTranscriptApi.list_transcripts(video_id))).fetch()
    return " ".join(d["text"] for d in data)


# ---------------------------------------------------------------- cleaning
def clean_text(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)          # [Music], [Applause]
    text = text.replace(">>", " ")
    text = re.sub(r"\b(?:um+|uh+|uhm|erm|hmm+|mhm)\b[,.]?", " ", text, flags=re.I)
    text = re.sub(r"\b([A-Za-z']+)(?:\s+\1\b)+", r"\1", text, flags=re.I)  # "I I I" -> "I"
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_words(text, chunk_words=1800, overlap=60):
    words = text.split()
    if len(words) <= chunk_words:
        return [text]
    chunks, step = [], chunk_words - overlap
    for i in range(0, len(words), step):
        chunks.append(" ".join(words[i:i + chunk_words]))
        if i + chunk_words >= len(words):
            break
    return chunks


# ---------------------------------------------------------------- model
def load_model(model_name="Qwen/Qwen2.5-3B-Instruct", load_in_4bit=False):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(model_name)
    kwargs = {"device_map": "auto"}
    if load_in_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        kwargs["dtype"] = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.eval()
    return tok, model


def chat(tok, model, user_prompt, max_new_tokens=600):
    import torch

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, repetition_penalty=1.05,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def count_tokens(tok, text):
    return len(tok(text, add_special_tokens=False)["input_ids"])


def translate_text(tok, model, text, language, max_new_tokens=900):
    prompt = (
        f"Translate the following text into {language}. Keep the exact same structure: "
        "keep the markdown headings (###) and bullet points, and translate the headings too. "
        "Keep names, numbers and game/product names unchanged. "
        "Output ONLY the translation, with no comments.\n\n"
        f"TEXT:\n{text}"
    )
    return chat(tok, model, prompt, max_new_tokens=max_new_tokens)

# ---------------------------------------------------------------- summarize
def summarize_transcript(text, tok, model, style="standard", language="English",
                         single_pass_tokens=6000, chunk_words=1800, progress=None):
    """Short videos: one pass. Long videos: extract notes per part, then merge."""
    say = progress or (lambda m: None)
    style_text = STYLE_INSTRUCTIONS[style]
    n_tokens = count_tokens(tok, text)
    notes = None

    if n_tokens <= single_pass_tokens:
        say(f"Transcript is {n_tokens} tokens: summarizing in one pass...")
        source_label = "TRANSCRIPT"
        content = text
        intro = "Summarize the following video transcript."
    else:
        chunks = split_words(text, chunk_words=chunk_words)
        notes = []
        for i, chunk in enumerate(chunks, 1):
            say(f"Reading part {i}/{len(chunks)} and extracting facts...")
            prompt = (
                f"This is part {i} of {len(chunks)} of an auto-generated transcript of ONE YouTube video.\n"
                "Extract the important facts as short bullet points: who is involved, what happens, "
                "rules or setup, claims, numbers (scores, counts, deaths, prices, dates) and any outcome, "
                "reveal or twist. Write 5-10 bullets. Do NOT add anything that is not in the text.\n\n"
                f"TRANSCRIPT PART {i}/{len(chunks)}:\n{chunk}"
            )
            notes.append(f"[Part {i}/{len(chunks)}]\n" + chat(tok, model, prompt, max_new_tokens=350))
        source_label = "NOTES FROM ALL PARTS OF THE VIDEO, IN ORDER"
        content = "\n\n".join(notes)
        intro = (
            "These notes were extracted from consecutive parts of ONE video transcript. "
            "Merge them into one summary of the whole video. Parts overlap slightly, so remove duplicates. "
            "Numbers from early parts may be intermediate values: prefer the final numbers from the last parts. "
            "The final outcome and any twist are usually in the LAST parts."
        )

    say("Writing the final summary...")
    prompt = (
        f"{intro}\n\n{style_text}\n\nWrite the answer in {language}. "
        "Do not mention any source, website or channel that is not in the text.\n\n"
        f"{source_label}:\n{content}"
    )
    summary = chat(tok, model, prompt, max_new_tokens=MAX_NEW_TOKENS[style])
    return {
        "summary": summary,
        "notes": notes,
        "n_tokens": n_tokens,
        "unsupported": find_unsupported(summary, text),
    }


# ---------------------------------------------------------------- grounding check
_NUM_WORDS = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
    "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13,
    "fourteen": 14, "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
    "nineteen": 19, "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60,
    "seventy": 70, "eighty": 80, "ninety": 90,
}
_TENS = {k: v for k, v in _NUM_WORDS.items() if v >= 20}
_UNITS = {k: v for k, v in _NUM_WORDS.items() if 1 <= v <= 9}


def _words_to_digits(text):
    t = text.lower()
    tens = "|".join(_TENS)
    units = "|".join(_UNITS)
    t = re.sub(rf"\b({tens})[- ]({units})\b", lambda m: str(_TENS[m.group(1)] + _UNITS[m.group(2)]), t)
    t = re.sub(r"\b(" + "|".join(_NUM_WORDS) + r")\b", lambda m: str(_NUM_WORDS[m.group(1)]), t)
    t = re.sub(r"\b(\d+) hundred\b", lambda m: str(int(m.group(1)) * 100), t)
    return t


def find_unsupported(summary, source):
    """Numbers and capitalized terms in the summary that never appear in the transcript."""
    body = "\n".join(l for l in summary.splitlines() if not l.strip().startswith("#"))
    src_norm = _words_to_digits(source)

    num_re = r"\d+(?:[.,]\d+)*"
    src_nums = {n.replace(",", "") for n in re.findall(num_re, src_norm)}
    numbers = sorted({
        n for n in re.findall(num_re, body)
        if n.replace(",", "") not in src_nums
    })

    src_words = set(re.findall(r"\w+", source.lower()))
    terms = set()
    for m in re.finditer(r"\b(?=\w*[A-Z])[A-Za-z]{3,}\b", body):
        before = body[:m.start()].rstrip(" \t")
        if before == "" or before[-1] in ".!?:-*\u2022\n":
            continue  # first word of a sentence/bullet
        if m.group(0).lower() not in src_words:
            terms.add(m.group(0))
    return {"numbers": numbers, "terms": sorted(terms)}

Writing summarizer.py


In [3]:
import importlib
import summarizer as S
importlib.reload(S)

MODEL_LABEL = "Qwen2.5-3B (recommended)"   # or "Qwen2.5-1.5B (fast, lower quality)" / "Qwen2.5-7B 4-bit (best quality)"
model_name, use_4bit = S.MODELS[MODEL_LABEL]
tok, model = S.load_model(model_name, use_4bit)
print("Loaded:", model_name)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct


# This cell is for testing

In [4]:
# VIDEO_URL = ""          # YouTube link
# TRANSCRIPT_TEXT = ""    # optional: paste a transcript here instead of fetching
# STYLE = "standard"      # short / standard / detailed
# OUTPUT_LANGUAGE = "English"   # e.g. "Arabic"

# if not VIDEO_URL and not TRANSCRIPT_TEXT:
#     print("Set VIDEO_URL or TRANSCRIPT_TEXT first.")
# else:
#     raw = TRANSCRIPT_TEXT or S.get_youtube_transcript(VIDEO_URL)
#     cleaned = S.clean_text(raw)
#     print(f"{len(raw.split())} words -> {len(cleaned.split())} after cleaning")
#     result = S.summarize_transcript(cleaned, tok, model, style=STYLE,
#                                     language=OUTPUT_LANGUAGE, progress=print)
#     print("\n" + "=" * 60)
#     print(result["summary"])
#     print("=" * 60)
#     if result["unsupported"]["numbers"] or result["unsupported"]["terms"]:
#         print("Double-check (not found in transcript):", result["unsupported"])

In [5]:
# Free GPU memory before launching Streamlit (it loads its own copy of the model)
import gc, torch
try:
    del model, tok
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed")

GPU memory freed


## Streamlit App

In [6]:
%%writefile app.py
import gc
import streamlit as st
import summarizer as S

st.set_page_config(page_title="YouTube Summarizer", page_icon="🎬", layout="centered")
st.title("🎬 YouTube Video Summarizer")

with st.sidebar:
    st.header("Settings")
    labels = list(S.MODELS)
    model_label = st.selectbox("Model", labels, index=labels.index(S.DEFAULT_MODEL_LABEL))
    style = st.selectbox("Summary style", ["short", "standard", "detailed"], index=1)
    out_lang = st.selectbox("Summary language", S.LANGUAGES)
    TRANSCRIPT_LANGS = {
        "English": ["en"],
        "Arabic": ["ar"],
    }
    transcript_choice = st.selectbox(
        "Video (transcript) language", list(TRANSCRIPT_LANGS),
        help="The language spoken in the video. If it is not available, any available transcript is used.",
    )


@st.cache_resource(show_spinner="Loading model (first time takes a few minutes)...")
def get_model(name, use_4bit):
    return S.load_model(name, use_4bit)


def load_selected(label):
    previous = st.session_state.get("loaded_model")
    if previous is not None and previous != label:
        st.cache_resource.clear()
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
    st.session_state["loaded_model"] = label
    name, use_4bit = S.MODELS[label]
    return get_model(name, use_4bit)


def clear_inputs():
    st.session_state["video_url"] = ""
    st.session_state["pasted_transcript"] = ""


url = st.text_input("YouTube video URL", placeholder="https://www.youtube.com/watch?v=...", key="video_url")
with st.expander("Or paste the transcript manually (useful if YouTube blocks the server)"):
    pasted = st.text_area("Transcript text", height=150, key="pasted_transcript")

col_summarize, col_clear = st.columns([3, 1])
summarize_clicked = col_summarize.button("Summarize", type="primary", use_container_width=True)
col_clear.button("Clear", on_click=clear_inputs, use_container_width=True)

if summarize_clicked:
    if not url.strip() and not pasted.strip():
        st.error("Paste a YouTube URL or a transcript first.")
    else:
        try:
            tok, model = load_selected(model_label)
            with st.status("Working...", expanded=True) as status:
                if pasted.strip():
                    raw = pasted
                else:
                    st.write("Fetching transcript...")
                    langs = TRANSCRIPT_LANGS[transcript_choice]
                    raw = S.get_youtube_transcript(url, languages=langs)
                cleaned = S.clean_text(raw)
                st.write(f"Transcript: {len(raw.split())} words -> {len(cleaned.split())} after cleaning")

                result = S.summarize_transcript(
                    cleaned, tok, model, style=style, language=out_lang,
                    progress=lambda m: st.write(m),
                )
                status.update(label="Done!", state="complete", expanded=False)

            vid = S.extract_video_id(url) if url.strip() else None
            if vid:
                st.video(f"https://www.youtube.com/watch?v={vid}")

            st.subheader("Summary")
            st.markdown(result["summary"])

            bad = result["unsupported"]
            if bad["numbers"] or bad["terms"]:
                st.warning(
                    "Please double-check these (not found in the transcript): "
                    + ", ".join(bad["numbers"] + bad["terms"])
                )

            st.download_button("Download summary (.txt)", result["summary"], file_name="summary.txt")
            if result["notes"]:
                with st.expander("Extracted notes per part"):
                    st.text("\n\n".join(result["notes"]))
            with st.expander("Cleaned transcript"):
                st.write(cleaned)
        except Exception as e:
            st.error(f"Something went wrong: {type(e).__name__}: {e}")

Writing app.py


In [7]:
import subprocess, sys, time, importlib.util

# install any missing package in this session
needed = {
    "streamlit": "streamlit",
    "youtube_transcript_api": "youtube-transcript-api",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
}
missing = [pip_name for mod, pip_name in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

subprocess.run("pkill -f 'streamlit run' || true", shell=True)
streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501",
     "--server.headless", "true", "--browser.gatherUsageStats", "false",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=open("streamlit_logs.txt", "w"), stderr=subprocess.STDOUT,
)
time.sleep(8)
if streamlit_proc.poll() is not None:
    print("Streamlit crashed. Logs:\n", open("streamlit_logs.txt").read())
else:
    print("Streamlit started on port 8501")

Streamlit started on port 8501


In [8]:
import subprocess, time, re, os

if not os.path.exists("cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared",
        shell=True, check=True,
    )

subprocess.run("pkill -f cloudflared || true", shell=True)
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=open("tunnel_logs.txt", "w"), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel_logs.txt").read())
    if m:
        public_url = m.group(0)
        break
print("Open your app here:\n", public_url or "Link not found, check tunnel_logs.txt")

Open your app here:
 https://somebody-hebrew-continuity-flow.trycloudflare.com
